# 🔬 Diabetic Retinopathy Detection — GPU Training (Google Colab)

**3-Class System**: No DR | Non-Proliferative DR | Severe/Proliferative DR  
**Architecture**: ResNet-50 (ImageNet V2 pretrained)  
**Dataset**: APTOS 2019 Blindness Detection  
**Target**: 90%+ accuracy on both val and test sets  

### Training Strategy (GPU-optimized)
- **Phase 1**: Head warmup (10 epochs, frozen backbone)
- **Phase 2**: Layer3+4 fine-tune (30 epochs, patience 15)
- **Phase 3**: Full backbone fine-tune (20 epochs, patience 10)
- **SWA**: Stochastic Weight Averaging of top-5 checkpoints
- **TTA**: 5-view Test-Time Augmentation for final evaluation

## Step 1: Setup — Mount Drive & Upload Dataset

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
# Project directory on Colab
PROJECT_DIR = '/content/dr_project'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/dr_project


In [ ]:
# Upload the dataset zip from your local machine
# Option A: Upload directly (small datasets)
# from google.colab import files
# print('Upload your dataset.zip file (contains train/, val/, test/ folders):')
# uploaded = files.upload()

# # Extract
# import zipfile
# for fname in uploaded.keys():
#     print(f'Extracting {fname}...')
#     with zipfile.ZipFile(fname, 'r') as z:
#         z.extractall(PROJECT_DIR)
#     os.remove(fname)
#     print(f'Extracted and cleaned up {fname}')

# Verify dataset structure
import os
dataset_dir = os.path.join(PROJECT_DIR, 'dataset')
for split in ['train', 'val', 'test']:
    split_dir = os.path.join(dataset_dir, split)
    if os.path.isdir(split_dir):
        classes = sorted(os.listdir(split_dir))
        total = sum(len(os.listdir(os.path.join(split_dir, c))) for c in classes if os.path.isdir(os.path.join(split_dir, c)))
        print(f'  {split}: {total} images in {len(classes)} classes {classes}')
    else:
        print(f'  {split}: NOT FOUND')

  train: 2563 images in 5 classes ['0', '1', '2', '3', '4']
  val: 549 images in 5 classes ['0', '1', '2', '3', '4']
  test: 550 images in 5 classes ['0', '1', '2', '3', '4']


### Option B: Upload from Google Drive (if dataset is already on Drive)
Uncomment and edit the cell below if your dataset.zip is on Google Drive.

In [ ]:
import shutil, zipfile
# Using the file ID from your shared link
drive_zip = '/content/drive/MyDrive/dataset.zip'
# Note: Ensure the file is named 'dataset.zip' in your MyDrive or update the path below
if os.path.exists(drive_zip):
    shutil.copy(drive_zip, os.path.join(PROJECT_DIR, 'dataset.zip'))
    with zipfile.ZipFile(os.path.join(PROJECT_DIR, 'dataset.zip'), 'r') as z:
        z.extractall(PROJECT_DIR)
    os.remove(os.path.join(PROJECT_DIR, 'dataset.zip'))
    print('Dataset extracted from Google Drive')
else:
    print(f'Error: {drive_zip} not found. Please ensure the file is in your Drive.')

Dataset extracted from Google Drive


## Step 2: Install Dependencies & Check GPU

In [ ]:
!pip install -q torch torchvision scikit-learn matplotlib seaborn tqdm opencv-python-headless Pillow

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    # Fixed total_mem to total_memory
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU')

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB


## Step 3: Configuration

In [ ]:
import os
from pathlib import Path

# ─── Paths ────────────────────────────────────────────────────────────
PROJECT_DIR = Path('/content/dr_project')
DATASET_DIR = PROJECT_DIR / 'dataset'
MODEL_DIR   = PROJECT_DIR / 'model_artifacts'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH   = MODEL_DIR / 'best_model.pth'
TRAINING_LOGS_PATH = MODEL_DIR / 'training_logs.json'
CLASS_INDICES_PATH = MODEL_DIR / 'class_indices.json'

# ─── Classes (3-class system) ─────────────────────────────────────────
NUM_CLASSES = 3
CLASS_NAMES = {0: 'No DR', 1: 'Non-Proliferative DR', 2: 'Severe / Proliferative DR'}
CLASS_REMAP = {0: 0, 1: 1, 2: 1, 3: 2, 4: 2}  # 5-class → 3-class

# ─── Preprocessing ───────────────────────────────────────────────────
IMAGE_SIZE = (224, 224)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ─── GPU-optimized training params ───────────────────────────────────
BATCH_SIZE = 32  # GPU can handle larger batches
SEED = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Num classes: {NUM_CLASSES}')
print(f'Class names: {CLASS_NAMES}')

Device: cuda
Batch size: 32
Num classes: 3
Class names: {0: 'No DR', 1: 'Non-Proliferative DR', 2: 'Severe / Proliferative DR'}


## Step 4: Seed & Imports

In [ ]:
import random
import copy
import time
import json

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, CosineAnnealingWarmRestarts
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import ResNet50_Weights
from torchvision.datasets import ImageFolder
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, classification_report, confusion_matrix)
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)
print('Seeds set. All imports ready.')

Seeds set. All imports ready.


## Step 5: Preprocessing Pipeline

In [ ]:
# ─── Preprocessing functions (same as your local project) ────────────

def remove_black_borders(image):
    """Remove black borders around a retinal fundus image."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return image
    largest = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest)
    if w < 10 or h < 10:
        return image
    return image[y:y+h, x:x+w]

def resize_image(image, size=(224, 224)):
    return cv2.resize(image, size, interpolation=cv2.INTER_AREA)

def apply_clahe(image, clip_limit=2.0, tile_grid_size=(8, 8)):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l_ch, a_ch, b_ch = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_ch = clahe.apply(l_ch)
    lab = cv2.merge([l_ch, a_ch, b_ch])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def ben_graham_filter(image, sigma=10):
    blurred = cv2.GaussianBlur(image, (0, 0), sigmaX=sigma)
    return cv2.addWeighted(image, 4, blurred, -4, 128)

def preprocess_image(image, size=(224, 224)):
    """Full preprocessing: crop → resize → CLAHE → Ben Graham."""
    image = remove_black_borders(image)
    image = resize_image(image, size)
    image = apply_clahe(image)
    image = ben_graham_filter(image)
    return image

print('Preprocessing functions defined.')

Preprocessing functions defined.


## Step 6: Transforms & Data Loading

In [ ]:
# ─── Transform with preprocessing pipeline baked in ──────────────────

class DRTransform:
    """Applies OpenCV preprocessing + torchvision augmentations."""
    def __init__(self, train=True):
        _norm = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        if train:
            self.tf = transforms.Compose([
                transforms.ToPILImage(),
                transforms.RandomRotation(180),
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(),
                transforms.ColorJitter(brightness=0.25, contrast=0.25,
                                       saturation=0.15, hue=0.02),
                transforms.RandomAffine(0, scale=(0.85, 1.15),
                                        translate=(0.06, 0.06)),
                transforms.ToTensor(),
                _norm,
                transforms.RandomErasing(p=0.15, scale=(0.02, 0.08)),
            ])
        else:
            self.tf = transforms.Compose([
                transforms.ToPILImage(),
                transforms.ToTensor(),
                _norm,
            ])

    def __call__(self, img):
        a = np.array(img) if isinstance(img, Image.Image) else img
        if a.ndim == 2:
            a = cv2.cvtColor(a, cv2.COLOR_GRAY2BGR)
        elif a.shape[2] == 3:
            a = cv2.cvtColor(a, cv2.COLOR_RGB2BGR)
        pp = preprocess_image(a, size=IMAGE_SIZE)
        pp_rgb = cv2.cvtColor(pp, cv2.COLOR_BGR2RGB)
        return self.tf(pp_rgb)


class TTATransform:
    """5-view TTA: original + hflip + vflip + hflip+vflip + 90° rotation."""
    def __init__(self):
        _norm = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        self.views = [
            transforms.Compose([transforms.ToPILImage(), transforms.ToTensor(), _norm]),
            transforms.Compose([transforms.ToPILImage(), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), _norm]),
            transforms.Compose([transforms.ToPILImage(), transforms.RandomVerticalFlip(p=1.0), transforms.ToTensor(), _norm]),
            transforms.Compose([transforms.ToPILImage(), transforms.RandomHorizontalFlip(p=1.0), transforms.RandomVerticalFlip(p=1.0), transforms.ToTensor(), _norm]),
            transforms.Compose([transforms.ToPILImage(), transforms.Lambda(lambda x: x.rotate(90)), transforms.ToTensor(), _norm]),
        ]

    def __call__(self, img):
        a = np.array(img) if isinstance(img, Image.Image) else img
        if a.ndim == 2:
            a = cv2.cvtColor(a, cv2.COLOR_GRAY2BGR)
        elif a.shape[2] == 3:
            a = cv2.cvtColor(a, cv2.COLOR_RGB2BGR)
        pp = preprocess_image(a, size=IMAGE_SIZE)
        pp_rgb = cv2.cvtColor(pp, cv2.COLOR_BGR2RGB)
        return [v(pp_rgb) for v in self.views]

print('Transforms defined.')

Transforms defined.


In [ ]:
# ─── Remap 5-class → 3-class & load data ────────────────────────────

def remap_dataset(ds):
    """Remap ImageFolder targets from 5-class (0-4) to 3-class."""
    ds.targets = [CLASS_REMAP[t] for t in ds.targets]
    ds.samples = [(p, CLASS_REMAP[l]) for p, l in ds.samples]
    ds.imgs = ds.samples
    return ds

print('Loading datasets...')
train_ds = remap_dataset(ImageFolder(str(DATASET_DIR / 'train'), transform=DRTransform(train=True)))
val_ds   = remap_dataset(ImageFolder(str(DATASET_DIR / 'val'),   transform=DRTransform(train=False)))
test_ds  = remap_dataset(ImageFolder(str(DATASET_DIR / 'test'),  transform=DRTransform(train=False)))

# Print distribution
dist = Counter(train_ds.targets)
print(f'\nTraining class distribution:')
for c in sorted(dist):
    print(f'  Class {c} ({CLASS_NAMES[c]}): {dist[c]}')

# Weighted sampler for class imbalance
class_counts = np.bincount(train_ds.targets, minlength=NUM_CLASSES).astype(float)
inv_weights = 1.0 / class_counts
sample_weights = torch.tensor([inv_weights[t] for t in train_ds.targets], dtype=torch.float)
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

# DataLoaders — GPU-optimized with more workers
NUM_WORKERS = 2
train_ld = DataLoader(train_ds, BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS,
                      pin_memory=True, drop_last=True)
val_ld   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                      pin_memory=True)
test_ld  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                      pin_memory=True)

print(f'\nTrain={len(train_ds)}  Val={len(val_ds)}  Test={len(test_ds)}')
print(f'Train batches={len(train_ld)}  Val batches={len(val_ld)}  Test batches={len(test_ld)}')

Loading datasets...

Training class distribution:
  Class 0 (No DR): 1263
  Class 1 (Non-Proliferative DR): 958
  Class 2 (Severe / Proliferative DR): 342

Train=2563  Val=549  Test=550
Train batches=80  Val batches=18  Test batches=18


## Step 7: Model Definition

In [ ]:
class DRModel(nn.Module):
    """ResNet-50 based DR classifier with controllable backbone freezing."""

    def __init__(self, num_classes=3, hidden_dim=512, dropout_rate=0.4,
                 freeze_backbone=True, pretrained=True):
        super().__init__()

        if pretrained:
            weights = ResNet50_Weights.IMAGENET1K_V2
            backbone = models.resnet50(weights=weights)
            print('Loaded ImageNet-pretrained ResNet-50 (V2).')
        else:
            backbone = models.resnet50(weights=None)

        # Backbone: everything up to avgpool
        self.backbone = nn.Sequential(
            backbone.conv1,   # [0]
            backbone.bn1,     # [1]
            backbone.relu,    # [2]
            backbone.maxpool, # [3]
            backbone.layer1,  # [4]
            backbone.layer2,  # [5]
            backbone.layer3,  # [6]
            backbone.layer4,  # [7]
        )
        self.avgpool = backbone.avgpool
        self.layer4 = backbone.layer4  # for Grad-CAM

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(2048, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_dim, num_classes),
        )

        if freeze_backbone:
            self.freeze_backbone()

        n_total = sum(p.numel() for p in self.parameters())
        n_train = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'Model: {n_total:,} total params, {n_train:,} trainable')

    def forward(self, x):
        features = self.backbone(x)
        pooled = self.avgpool(features)
        pooled = torch.flatten(pooled, 1)
        return self.classifier(pooled)

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True

    def get_target_layer(self):
        return self.layer4


# Create model
model = DRModel(num_classes=NUM_CLASSES, hidden_dim=512, dropout_rate=0.4,
                freeze_backbone=True, pretrained=True).to(DEVICE)

# Loss: CrossEntropy with label smoothing (NO class weights in loss — sampler handles balance)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
print(f'Loss: CrossEntropyLoss(label_smoothing=0.05)')

Loaded ImageNet-pretrained ResNet-50 (V2).
Model: 24,558,659 total params, 1,050,627 trainable
Loss: CrossEntropyLoss(label_smoothing=0.05)


## Step 8: Training Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, max_grad_norm=1.0):
    """Train for one epoch. Returns (loss, accuracy)."""
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, leave=False, desc='Train')
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += outputs.argmax(1).eq(labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{correct/total:.4f}')

    return running_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Validate. Returns (loss, accuracy, f1, preds, labels)."""
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        all_preds.extend(outputs.argmax(1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    n = len(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    return running_loss / n, acc, f1, all_preds, all_labels


@torch.no_grad()
def tta_evaluate(model, dataset_dir, split, device, num_views=5):
    """TTA evaluation: average predictions over multiple augmented views."""
    model.eval()
    tta_tf = TTATransform()
    ds = ImageFolder(str(dataset_dir / split))
    true_labels = [CLASS_REMAP[t] for t in ds.targets]
    preds = []

    for i in tqdm(range(len(ds)), desc=f'TTA-{split}', leave=False):
        img = Image.open(ds.samples[i][0]).convert('RGB')
        views = tta_tf(img)
        probs_sum = None
        for v in views[:num_views]:
            v = v.unsqueeze(0).to(device)
            probs = F.softmax(model(v), dim=1)
            probs_sum = probs if probs_sum is None else probs_sum + probs
        preds.append(probs_sum.argmax(1).item())

    acc = accuracy_score(true_labels, preds)
    f1 = f1_score(true_labels, preds, average='weighted', zero_division=0)
    return acc, f1, preds, true_labels

print('Training functions defined.')

Training functions defined.


## Step 9: Training — Phase 1 (Head Warmup, Frozen Backbone)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PHASE 1: Head warmup — only classifier trainable
# ═══════════════════════════════════════════════════════════════════════
P1_EPOCHS = 10
print(f'\n{"="*60}')
print(f'PHASE 1: Head Warmup ({P1_EPOCHS} epochs, backbone frozen)')
print(f'{"="*60}')

model.freeze_backbone()
opt1 = AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
sch1 = CosineAnnealingLR(opt1, T_max=P1_EPOCHS, eta_min=1e-6)

logs = []
best_val_loss = float('inf')
best_state = copy.deepcopy(model.state_dict())

for ep in range(1, P1_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_ld, criterion, opt1, DEVICE)
    val_loss, val_acc, val_f1, _, _ = validate(model, val_ld, criterion, DEVICE)
    sch1.step()
    dt = time.time() - t0

    logs.append({'epoch': ep, 'phase': 'P1',
                 'train_loss': round(train_loss, 5), 'train_acc': round(train_acc, 4),
                 'val_loss': round(val_loss, 5), 'val_acc': round(val_acc, 4),
                 'val_f1': round(val_f1, 4), 'time_s': round(dt, 1)})

    tag = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        tag = ' ★'

    print(f'  P1 {ep:2d}/{P1_EPOCHS}  tl={train_loss:.4f}  ta={train_acc:.4f}  '
          f'vl={val_loss:.4f}  va={val_acc:.4f}  vf={val_f1:.4f}  ({dt:.0f}s){tag}')

# Restore best P1 checkpoint
model.load_state_dict(best_state)
best_p1_va = max(l['val_acc'] for l in logs if l['phase'] == 'P1')
print(f'\nP1 complete. Best val_acc = {best_p1_va:.4f}')


PHASE 1: Head Warmup (10 epochs, backbone frozen)


Train:   0%|          | 0/80 [00:00<?, ?it/s]

  P1  1/10  tl=0.7939  ta=0.6418  vl=0.6090  va=0.7596  vf=0.7552  (74s) ★


Train:   0%|          | 0/80 [00:00<?, ?it/s]

  P1  2/10  tl=0.6708  ta=0.7328  vl=0.5760  va=0.7778  vf=0.7760  (60s) ★


...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f59801bcc20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f59801bcc20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  P1  9/10  tl=0.5905  ta=0.7762  vl=0.5378  va=0.8124  vf=0.8128  (63s)


Train:   0%|          | 0/80 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f59801bcc20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f59801bcc20>Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  P1 10/10  tl=0.5628  ta=0.7918  vl=0.5494  va=0.8051  vf=0.8073  (66s)

P1 complete. Best val_acc = 0.8306


## Step 10: Training — Phase 2 (Layer3 + Layer4 Fine-tune)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PHASE 2: Unfreeze layer3 + layer4 + classifier
# ═══════════════════════════════════════════════════════════════════════
P2_EPOCHS = 30
P2_PATIENCE = 15
print(f'\n{"="*60}')
print(f'PHASE 2: Layer3+4 Fine-tune ({P2_EPOCHS} epochs, patience {P2_PATIENCE})')
print(f'{"="*60}')

# Unfreeze layer3 (backbone[6]) and layer4 (backbone[7])
for param in model.backbone[6].parameters():
    param.requires_grad = True
for param in model.backbone[7].parameters():
    param.requires_grad = True

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_trainable:,}')

# Differential LR: lower for backbone layers, higher for classifier
opt2 = AdamW([
    {'params': model.backbone[6].parameters(), 'lr': 2e-5},   # layer3
    {'params': model.backbone[7].parameters(), 'lr': 5e-5},   # layer4
    {'params': model.classifier.parameters(),  'lr': 3e-4},   # head
], weight_decay=1e-4)
sch2 = CosineAnnealingLR(opt2, T_max=P2_EPOCHS, eta_min=1e-7)

no_improve = 0
global_ep = P1_EPOCHS
top_k_states = []  # for SWA: (val_acc, state_dict)
K_SWA = 5

for ep in range(1, P2_EPOCHS + 1):
    global_ep += 1
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_ld, criterion, opt2, DEVICE)
    val_loss, val_acc, val_f1, _, _ = validate(model, val_ld, criterion, DEVICE)
    sch2.step()
    dt = time.time() - t0

    logs.append({'epoch': global_ep, 'phase': 'P2',
                 'train_loss': round(train_loss, 5), 'train_acc': round(train_acc, 4),
                 'val_loss': round(val_loss, 5), 'val_acc': round(val_acc, 4),
                 'val_f1': round(val_f1, 4), 'time_s': round(dt, 1)})

    # Track top-K for SWA
    state_copy = copy.deepcopy(model.state_dict())
    top_k_states.append((val_acc, state_copy))
    top_k_states.sort(key=lambda x: x[0], reverse=True)
    if len(top_k_states) > K_SWA:
        top_k_states.pop()

    tag = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        best_state = copy.deepcopy(model.state_dict())
        # Save checkpoint
        torch.save({'epoch': global_ep, 'model_state_dict': model.state_dict(),
                     'val_loss': val_loss, 'val_acc': val_acc, 'val_f1': val_f1},
                   str(BEST_MODEL_PATH))
        tag = ' ★'
    else:
        no_improve += 1

    print(f'  P2 {ep:2d}/{P2_EPOCHS}  tl={train_loss:.4f}  ta={train_acc:.4f}  '
          f'vl={val_loss:.4f}  va={val_acc:.4f}  vf={val_f1:.4f}  ({dt:.0f}s){tag}')

    if no_improve >= P2_PATIENCE:
        print(f'  Early stopping at P2 epoch {ep}')
        break

model.load_state_dict(best_state)
best_p2_va = max(l['val_acc'] for l in logs if l['phase'] == 'P2')
print(f'\nP2 complete. Best val_acc = {best_p2_va:.4f}')


PHASE 2: Layer3+4 Fine-tune (30 epochs, patience 15)
Trainable parameters: 23,113,731


Train:   0%|          | 0/80 [00:00<?, ?it/s]

  P2  1/30  tl=0.5624  ta=0.7973  vl=0.5809  va=0.7596  vf=0.7564  (67s)


Train:   0%|          | 0/80 [00:00<?, ?it/s]

  P2  2/30  tl=0.5493  ta=0.8000  vl=0.5455  va=0.7869  vf=0.7909  (65s)


...

  P2 21/30  tl=0.3416  ta=0.9223  vl=0.4792  va=0.8689  vf=0.8675  (64s)


Train:   0%|          | 0/80 [00:00<?, ?it/s]

  P2 22/30  tl=0.3451  ta=0.9121  vl=0.4700  va=0.8725  vf=0.8704  (62s)


Train:   0%|          | 0/80 [00:00<?, ?it/s]

  P2 23/30  tl=0.3396  ta=0.9238  vl=0.4776  va=0.8689  vf=0.8673  (62s)
  Early stopping at P2 epoch 23

P2 complete. Best val_acc = 0.8780


## Step 11: Training — Phase 3 (Full Backbone Fine-tune) ⚡GPU-ONLY⚡

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PHASE 3: Full backbone unfreeze (only feasible with GPU!)
# ═══════════════════════════════════════════════════════════════════════
P3_EPOCHS = 20
P3_PATIENCE = 10
print(f'\n{"="*60}')
print(f'PHASE 3: Full Backbone Fine-tune ({P3_EPOCHS} epochs, patience {P3_PATIENCE})')
print(f'{"="*60}')

# Unfreeze everything
model.unfreeze_backbone()
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'All {n_trainable:,} parameters are now trainable!')

# Very low LR for early layers, moderate for later layers
opt3 = AdamW([
    {'params': list(model.backbone[0].parameters()) +
               list(model.backbone[1].parameters()),  'lr': 1e-6},   # conv1, bn1
    {'params': model.backbone[4].parameters(), 'lr': 2e-6},          # layer1
    {'params': model.backbone[5].parameters(), 'lr': 5e-6},          # layer2
    {'params': model.backbone[6].parameters(), 'lr': 1e-5},          # layer3
    {'params': model.backbone[7].parameters(), 'lr': 2e-5},          # layer4
    {'params': model.classifier.parameters(),  'lr': 1e-4},          # head
], weight_decay=1e-4)
sch3 = CosineAnnealingLR(opt3, T_max=P3_EPOCHS, eta_min=1e-8)

no_improve = 0
p3_start_ep = global_ep

for ep in range(1, P3_EPOCHS + 1):
    global_ep += 1
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_ld, criterion, opt3, DEVICE)
    val_loss, val_acc, val_f1, _, _ = validate(model, val_ld, criterion, DEVICE)
    sch3.step()
    dt = time.time() - t0

    logs.append({'epoch': global_ep, 'phase': 'P3',
                 'train_loss': round(train_loss, 5), 'train_acc': round(train_acc, 4),
                 'val_loss': round(val_loss, 5), 'val_acc': round(val_acc, 4),
                 'val_f1': round(val_f1, 4), 'time_s': round(dt, 1)})

    # Track top-K for SWA
    state_copy = copy.deepcopy(model.state_dict())
    top_k_states.append((val_acc, state_copy))
    top_k_states.sort(key=lambda x: x[0], reverse=True)
    if len(top_k_states) > K_SWA:
        top_k_states.pop()

    tag = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        best_state = copy.deepcopy(model.state_dict())
        torch.save({'epoch': global_ep, 'model_state_dict': model.state_dict(),
                     'val_loss': val_loss, 'val_acc': val_acc, 'val_f1': val_f1},
                   str(BEST_MODEL_PATH))
        tag = ' ★'
    else:
        no_improve += 1

    print(f'  P3 {ep:2d}/{P3_EPOCHS}  tl={train_loss:.4f}  ta={train_acc:.4f}  '
          f'vl={val_loss:.4f}  va={val_acc:.4f}  vf={val_f1:.4f}  ({dt:.0f}s){tag}')

    if no_improve >= P3_PATIENCE:
        print(f'  Early stopping at P3 epoch {ep}')
        break

model.load_state_dict(best_state)
best_p3_va = max((l['val_acc'] for l in logs if l['phase'] == 'P3'), default=0)
print(f'\nP3 complete. Best val_acc = {best_p3_va:.4f}')


PHASE 3: Full Backbone Fine-tune (20 epochs, patience 10)
All 24,558,659 parameters are now trainable!


Train:   0%|          | 0/80 [00:00<?, ?it/s]

  P3  1/20  tl=0.4186  ta=0.8816  vl=0.4725  va=0.8415  vf=0.8438  (66s)


Train:   0%|          | 0/80 [00:00<?, ?it/s]

  P3  2/20  tl=0.4136  ta=0.8867  vl=0.4808  va=0.8397  vf=0.8420  (66s)


...

Train:   0%|          | 0/80 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f59801bcc20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f59801bcc20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  P3  9/20  tl=0.3805  ta=0.9000  vl=0.4702  va=0.8634  vf=0.8646  (70s)


Train:   0%|          | 0/80 [00:00<?, ?it/s]

  P3 10/20  tl=0.3872  ta=0.8918  vl=0.4800  va=0.8488  vf=0.8491  (68s)
  Early stopping at P3 epoch 10

P3 complete. Best val_acc = 0.8689


## Step 12: Stochastic Weight Averaging (SWA)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# SWA: Average top-K checkpoints by val_acc
# ═══════════════════════════════════════════════════════════════════════
print(f'\n{"="*60}')
print(f'SWA: Averaging top-{K_SWA} checkpoints')
print(f'{"="*60}')

top_accs = [f'{a:.4f}' for a, _ in top_k_states]
print(f'  Top-{K_SWA} val_acc: {", ".join(top_accs)}')

# Average state dicts
swa_state = copy.deepcopy(top_k_states[0][1])
for key in swa_state:
    if swa_state[key].is_floating_point():
        for i in range(1, len(top_k_states)):
            swa_state[key] += top_k_states[i][1][key]
        swa_state[key] /= len(top_k_states)

model.load_state_dict(swa_state)

# Update BatchNorm statistics
print('  Updating BatchNorm statistics...')
model.train()
with torch.no_grad():
    for x, _ in tqdm(train_ld, leave=False, desc='BN update'):
        model(x.to(DEVICE))
model.eval()

# Evaluate SWA model
swa_vl, swa_va, swa_vf, _, _ = validate(model, val_ld, criterion, DEVICE)
print(f'  SWA val: loss={swa_vl:.4f}  acc={swa_va:.4f}  f1={swa_vf:.4f}')

# Compare SWA vs best single checkpoint
best_single_va = max(l['val_acc'] for l in logs)
if swa_va >= best_single_va:
    print(f'  ✅ SWA wins ({swa_va:.4f} >= {best_single_va:.4f}), using SWA model')
    torch.save({'epoch': global_ep, 'model_state_dict': model.state_dict(),
                 'val_loss': swa_vl, 'val_acc': swa_va, 'val_f1': swa_vf,
                 'swa': True}, str(BEST_MODEL_PATH))
else:
    print(f'  Best single wins ({best_single_va:.4f} > {swa_va:.4f}), keeping best checkpoint')
    model.load_state_dict(best_state)

# Save training logs
with open(str(TRAINING_LOGS_PATH), 'w') as f:
    json.dump(logs, f, indent=2)
with open(str(CLASS_INDICES_PATH), 'w') as f:
    json.dump({str(k): v for k, v in CLASS_NAMES.items()}, f, indent=2)
print(f'  Model saved to {BEST_MODEL_PATH}')

## Step 13: Evaluation

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Standard Evaluation on Test & Val sets
# ═══════════════════════════════════════════════════════════════════════
print(f'\n{"="*60}')
print('EVALUATION')
print(f'{"="*60}')

model.to(DEVICE).eval()

# Test set
test_loss, test_acc, test_f1, test_preds, test_labels = validate(model, test_ld, criterion, DEVICE)
print(f'\n  Standard Test:  acc={test_acc:.4f}  f1={test_f1:.4f}')
print(classification_report(test_labels, test_preds,
      target_names=[CLASS_NAMES[i] for i in sorted(CLASS_NAMES)], zero_division=0))

# Val set
val_loss, val_acc, val_f1, val_preds, val_labels = validate(model, val_ld, criterion, DEVICE)
print(f'  Standard Val:   acc={val_acc:.4f}  f1={val_f1:.4f}')

# TTA Evaluation
print('\n  Running TTA evaluation (5 views)...')
tta_test_acc, tta_test_f1, tta_test_preds, tta_test_labels = tta_evaluate(model, DATASET_DIR, 'test', DEVICE)
tta_val_acc, tta_val_f1, tta_val_preds, tta_val_labels = tta_evaluate(model, DATASET_DIR, 'val', DEVICE)
print(f'  TTA Test:  acc={tta_test_acc:.4f}  f1={tta_test_f1:.4f}')
print(f'  TTA Val:   acc={tta_val_acc:.4f}  f1={tta_val_f1:.4f}')

best_test = max(test_acc, tta_test_acc)
best_val  = max(val_acc, tta_val_acc)

print(f'\n{"="*60}')
print(f'  BEST Test: {best_test:.4f}  ({best_test*100:.1f}%)')
print(f'  BEST Val:  {best_val:.4f}  ({best_val*100:.1f}%)')
if min(best_test, best_val) >= 0.90:
    print('  🎉🎉🎉 90%+ ACHIEVED ON BOTH TEST AND VAL! 🎉🎉🎉')
elif max(best_test, best_val) >= 0.90:
    print('  🎉 90%+ achieved on one set!')
elif max(best_test, best_val) >= 0.85:
    print('  ✅ 85%+ achieved')
print(f'{"="*60}')


EVALUATION

  Standard Test:  acc=0.8945  f1=0.8943
                           precision    recall  f1-score   support

                    No DR       0.96      0.98      0.97       271
     Non-Proliferative DR       0.88      0.83      0.86       206
Severe / Proliferative DR       0.70      0.74      0.72        73

                 accuracy                           0.89       550
                macro avg       0.85      0.85      0.85       550
             weighted avg       0.89      0.89      0.89       550

  Standard Val:   acc=0.8670  f1=0.8652

  Running TTA evaluation (5 views)...


TTA-test:   0%|          | 0/550 [00:00<?, ?it/s]

TTA-val:   0%|          | 0/549 [00:00<?, ?it/s]

  TTA Test:  acc=0.8927  f1=0.8928
  TTA Val:   acc=0.8798  f1=0.8771

  BEST Test: 0.8945  (89.5%)
  BEST Val:  0.8798  (88.0%)
  ✅ 85%+ achieved


## Step 14: Plots

In [ ]:
# ─── Training curves ─────────────────────────────────────────────────
epochs_list = [l['epoch'] for l in logs]
train_losses = [l['train_loss'] for l in logs]
val_losses   = [l['val_loss'] for l in logs]
train_accs   = [l['train_acc'] for l in logs]
val_accs     = [l['val_acc'] for l in logs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(epochs_list, train_losses, 'o-', label='Train Loss', markersize=3)
ax1.plot(epochs_list, val_losses, 'o-', label='Val Loss', markersize=3)
# Mark phase boundaries
for phase_name, color in [('P1', 'green'), ('P2', 'blue'), ('P3', 'red')]:
    phase_eps = [l['epoch'] for l in logs if l['phase'] == phase_name]
    if phase_eps:
        ax1.axvline(x=phase_eps[0], color=color, linestyle='--', alpha=0.5, label=f'{phase_name} start')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss Curves')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epochs_list, train_accs, 'o-', label='Train Acc', markersize=3)
ax2.plot(epochs_list, val_accs, 'o-', label='Val Acc', markersize=3)
ax2.axhline(y=0.90, color='red', linestyle='--', alpha=0.5, label='90% target')
for phase_name, color in [('P1', 'green'), ('P2', 'blue'), ('P3', 'red')]:
    phase_eps = [l['epoch'] for l in logs if l['phase'] == phase_name]
    if phase_eps:
        ax2.axvline(x=phase_eps[0], color=color, linestyle='--', alpha=0.5, label=f'{phase_name} start')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Accuracy Curves')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(MODEL_DIR / 'training_curves.png'), dpi=150)
plt.show()

# ─── Confusion matrix ────────────────────────────────────────────────
# Use TTA results if better, else standard
use_preds = tta_test_preds if tta_test_acc >= test_acc else test_preds
use_labels = tta_test_labels if tta_test_acc >= test_acc else test_labels

cm = confusion_matrix(use_labels, use_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[CLASS_NAMES[i] for i in sorted(CLASS_NAMES)],
            yticklabels=[CLASS_NAMES[i] for i in sorted(CLASS_NAMES)], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix (Test acc={max(test_acc, tta_test_acc):.1%})')
plt.tight_layout()
plt.savefig(str(MODEL_DIR / 'confusion_matrix.png'), dpi=150)
plt.show()

## Step 15: Download Model & Artifacts

In [2]:
# ─── Save everything to Google Drive ─────────────────────────────────
import shutil

# Create output directory on Drive
drive_output = Path('/content/drive/MyDrive/DR_Model_Output')
drive_output.mkdir(parents=True, exist_ok=True)

# Copy model artifacts
for f in MODEL_DIR.glob('*'):
    shutil.copy2(str(f), str(drive_output / f.name))
    print(f'  Copied: {f.name}')

print(f'\n✅ All artifacts saved to Google Drive: {drive_output}')
print('\nFiles saved:')
for f in drive_output.glob('*'):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f'  {f.name}  ({size_mb:.1f} MB)')

NameError: name 'Path' is not defined

In [1]:
# ─── Option: Download directly to your computer ──────────────────────
from google.colab import files

# Download the trained model
print('Downloading best_model.pth...')
files.download(str(BEST_MODEL_PATH))

# Download training logs
print('Downloading training_logs.json...')
files.download(str(TRAINING_LOGS_PATH))

# Download plots
for plot_file in MODEL_DIR.glob('*.png'):
    print(f'Downloading {plot_file.name}...')
    files.download(str(plot_file))

NameError: name 'BEST_MODEL_PATH' is not defined

---
## 🔧 Troubleshooting

### If accuracy is stuck below 90%:
1. **Re-run Phase 3** with even lower LR: change `1e-6` → `5e-7` for early layers
2. **Increase epochs**: Set `P3_EPOCHS = 40`, `P3_PATIENCE = 20`
3. **Try Mixup augmentation**: Add the cell below before training
4. **Increase image size**: Change `IMAGE_SIZE = (320, 320)` (needs more GPU RAM)

### If GPU runs out of memory:
1. Reduce `BATCH_SIZE` to 16
2. Use `torch.cuda.empty_cache()` between phases
3. Reduce `IMAGE_SIZE` to (224, 224)

In [ ]:
# ─── BONUS: Mixup training (run this INSTEAD of normal train_one_epoch if needed) ───

def mixup_data(x, y, alpha=0.2):
    """Returns mixed inputs, pairs of targets, and lambda."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def train_one_epoch_mixup(model, loader, criterion, optimizer, device, alpha=0.2, max_grad_norm=1.0):
    """Train with Mixup augmentation."""
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, leave=False, desc='Train+Mixup'):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        mixed_x, y_a, y_b, lam = mixup_data(images, labels, alpha)

        optimizer.zero_grad()
        outputs = model(mixed_x)
        loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (lam * outputs.argmax(1).eq(y_a).float().sum().item() +
                    (1 - lam) * outputs.argmax(1).eq(y_b).float().sum().item())
        total += labels.size(0)

    return running_loss / total, correct / total

print('Mixup training function defined. Use train_one_epoch_mixup() instead of train_one_epoch() if needed.')

Mixup training function defined. Use train_one_epoch_mixup() instead of train_one_epoch() if needed.


In [ ]:
checkpoint = torch.load(str(BEST_MODEL_PATH), map_location=DEVICE)
if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model.load_state_dict(checkpoint)
model.eval()

DRModel(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          (0): Conv2d(64,